# Umbrella Sampling

Umbrella sampling is a technique that allows for the sampling of structures that would normally be rare events in standard molecular simulations, such as unstable high-energy states (like transition states).
This is achieved by applying an artificial "Umbrella" potential to keep the system at a desired location along a reaction coordinate.

In the previous steps, we prepared the initial structures along the reaction coordinate (end-to-end distance).
In this notebook, we will perform MD simulations for each of these initial structures while applying the umbrella potential.

## Step 1. Importing Libraries and Environment Setup
We will load the libraries necessary for MD simulations and `joblib` for parallel computing.
Additionally, we will set the environment variables required to call PLUMED from Python.

In [ ]:
import os
import sys
import numpy as np
from time import perf_counter
from joblib import Parallel, delayed

# ASE
from ase import units
from ase.io import read, write
from ase.md.langevin import Langevin
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution, Stationary
# PLUMED wrapper
from ase.calculators.plumed import Plumed

# PFP (Matlantis)
from pfp_api_client.pfp.calculators.ase_calculator import ASECalculator
from pfp_api_client.pfp.estimator import Estimator

# PLUMED Environment Variables (Please modify if necessary to match the path of your environment.)
plumed_path = "/home/jovyan/local/plumed-2.9.0"
os.environ["PLUMED_KERNEL"] = f"{plumed_path}/lib/libplumedKernel.so"
os.environ["PLUMED_TYPESAFE_IGNORE"] = "yes"
sys.path.append(plumed_path)

## Step 2. Setting Calculation Parameters

Set the simulation temperature, duration, and the range of the reaction coordinate for umbrella sampling.
The number of parallel jobs for joblib is also specified here.

* **N_JOBS**: The number of calculations to run simultaneously.
* **colvars**: A list of target distances for umbrella sampling (e.g., 13.0Å, 13.5Å, ...).

In [ ]:
# PFP Settings
CALC_MODE     = "R2SCAN_PLUS_D3"
METHOD_TYPE   = "PFVM_D3_PFVM"
MODEL_VERSION = "v8.0.0"

# Joblib Settings
N_JOBS  = 10           # 並列数
VERBOSE = 10           # 進捗表示レベル
BACKEND = "threading"  # Matlantis利用時は "threading" 推奨

# MD Settings
TEMPERATURE  = 300.0    # Kelvin
TIMESTEP     = 1.0 * units.fs
TOTAL_STEPS  = 200_000
LOG_INTERVAL = 500

# Reaction Coordinate (Umbrella Windows)
colvars = np.arange(13.0, 32.01, 0.5)

## Step 3. Defining the Function for a Single Window

To parallelize the simulations across different umbrella windows using `joblib`, we define a function that executes the MD for a single target distance (window).

**Key Points for PLUMED Settings (plumed_setting):**
* The unit system is specified in `UNITS` (Energy: eV, Distance: Å).
* `DISTANCE ATOMS=9,99`: Specifies the indices of the atoms to be pulled (Note: PLUMED is 1-based, so ASE indices 8 and 98 become 9 and 99).
* `RESTRAINT ... KAPPA=0.2 AT={cv_str}`: Constrains the system to the target distance `AT` using a spring constant `KAPPA` of 0.2.

In [ ]:
def run_us_window(cv):
    """
    Function to execute umbrella sampling at a specified reaction coordinate cv (distance).
    """
    s_time = perf_counter()

    # --------------------------------------------------------
    # 1. Prepare Variables & Directories
    # --------------------------------------------------------
    cv_str = f"{cv:.2f}"
    out_dir = f"./output/04_umbrella_sampling/cv_{cv_str}"

    # Create directory before starting calculation
    os.makedirs(out_dir, exist_ok=True)

    # --------------------------------------------------------
    # 2. Prepare Calculator (Instantiate inside function)
    # --------------------------------------------------------
    estimator = Estimator(
        calc_mode=CALC_MODE,
        method_type=METHOD_TYPE,
        model_version=MODEL_VERSION
    )
    calculator = ASECalculator(estimator)

    # --------------------------------------------------------
    # 3. Prepare Atoms
    # --------------------------------------------------------
    input_xyz = f'./input/04_umbrella_sampling/initial_cv_{cv_str}.xyz'
    if not os.path.exists(input_xyz):
        input_xyz = f'./assets/04_umbrella_sampling/initial_cv_{cv_str}.xyz'

    atoms = read(input_xyz)

    # --------------------------------------------------------
    # 4. PLUMED Settings
    # --------------------------------------------------------
    # Umbrella potential settings
    # 1. Define interatomic distance (dist) (index is 1-based)
    # 2. Constrain with harmonic potential (RESTRAINT): V(x) = 0.5 * KAPPA * (x - AT)^2
    plumed_setting = [f"UNITS LENGTH=A ENERGY=eV",
        # 1. Define distance
        "dist: DISTANCE ATOMS=9,99",

        # 2. Apply umbrella potential (https://www.plumed.org/doc-v2.9/user-doc/html/lugano-2.html)
        #    1eV = 96.48 kJ/mol
        f"restraint: RESTRAINT ARG=dist KAPPA=0.2 AT={cv_str}",

        # 3. Output
        f"PRINT STRIDE=500 ARG=dist,restraint.bias,restraint.force2 FILE={out_dir}/COLVAR_{cv_str}",

        # Force buffer flush every 1000 steps
        "FLUSH STRIDE=1000"
    ]

    # Attach PLUMED calculator
    atoms.calc = Plumed(
        calc=calculator,
        input=plumed_setting,
        timestep=TIMESTEP,
        atoms=atoms,
        kT=units.kB * TEMPERATURE
    )

    # --------------------------------------------------------
    # 5. MD Simulation Setup
    # --------------------------------------------------------

    MaxwellBoltzmannDistribution(atoms, temperature_K=TEMPERATURE, force_temp=True)
    Stationary(atoms)

    # Dynamics
    dyn = Langevin(
        atoms,
        TIMESTEP,
        temperature_K=TEMPERATURE,
        friction=0.002/units.fs,
        trajectory=f'{out_dir}/md-dyn.traj',
        logfile=f'{out_dir}/md-dyn.log',
        loginterval=LOG_INTERVAL
    )

    # --------------------------------------------------------
    # 6. Run Execution
    # --------------------------------------------------------
    dyn.run(TOTAL_STEPS)

    # Output Restart
    write(f'{out_dir}/md-dyn-restart.pdb', atoms)

    elapsed_time = perf_counter() - s_time
    return f"Done: cv={cv_str} ({elapsed_time:.1f} sec)"

## Step 4. Running Umbrella Sampling

Using joblib.Parallel, we will execute the calculations for all configured windows in parallel.

In [ ]:
# ============================================================
# 4. Main Execution
# ============================================================

print(f"Start Parallel Calculation: {len(colvars)} windows")
print(f"Settings: n_jobs={N_JOBS}, backend={BACKEND}")

# Parallel execution
results = Parallel(n_jobs=N_JOBS, verbose=VERBOSE, backend=BACKEND)(
    delayed(run_us_window)(cv) for cv in colvars
)

In [ ]:
# Show results
print("\n--- Results ---")
for res in results:
    print(res)

## Notes

In this notebook, we performed data collection via umbrella sampling.
For accurate free energy calculations, it is crucial that there is sufficient overlap between the histograms of adjacent windows along the reaction coordinate.
If the overlap is insufficient, it may cause errors during analysis or prevent the calculation from converging.
In such cases, you should either change the spring constant KAPPA or add new windows between the existing ones.


## Next Step
In [the next notebook](./06_mbar_free_energy_en.ipynb), we will proceed to the free energy calculation.